In [1]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../Dataset/powerplant_data.csv")

In [3]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [4]:
# AT => temperature
# V => vacuum
# AP => pressure
# RH => humidity

# PE => produced energy

In [5]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [6]:
X = df.drop("PE", axis = 1)
y = df['PE']

In [7]:
# Split Our data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
X_train.shape

(7654, 4)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [10]:
import torch
import torch.nn as nn

In [11]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

In [12]:
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [13]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [14]:
train_loder = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loder = DataLoader(test_dataset, batch_size=32)

In [15]:
train_dataset[0]

(tensor([ 0.7481,  0.7201, -0.3266, -0.4971]), tensor([442.7500]))

### Deep Learning

In [16]:
## Define Our ANN Model

In [17]:
class ANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            # 2nd hidden Layer
            nn.Linear(6, 6),
            nn.ReLU(),

            # Output Layer
            nn.Linear(6, 1)
        )

    def forward(self, x):
        return self.model(x)

In [18]:
import torch.optim as optim

model = ANN()

# Loss, Optimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [19]:
type(model)

__main__.ANN

In [20]:
model

ANN(
  (model): Sequential(
    (0): Linear(in_features=4, out_features=6, bias=True)
    (1): ReLU()
    (2): Linear(in_features=6, out_features=6, bias=True)
    (3): ReLU()
    (4): Linear(in_features=6, out_features=1, bias=True)
  )
)

## Train ANN

In [21]:
train_loses = []
valid_loses = []
epochs = 100

for epochs in range(epochs):
    model.train()
    running_lose = 0.0  # Save the total Training Loss for 1 epoch

    
    for xb, yb in train_loder:
        # xb = features of 1st batch
        # yb = features of 1st bath
        optimizer.zero_grad()
        
        outputs = model(xb) # forword prop... Predicted Outputs for this batch
        loss = crietrion(outputs, yb) # Compute loss
        loss.backward() # back prop... Compute gradients
        optimizer.step() # Params Update
        
        running_lose += loss.item() # loss is a tensor ==> py float
        
    epoch_train_loss = running_lose / len(train_loder)
    train_loses.append(epoch_train_loss)

epoch 1/0 ===> train loss = 204961.725390625 & val loss = 200990.671875
epoch 2/1 ===> train loss = 189732.50970052084 & val loss = 172916.484375
epoch 3/2 ===> train loss = 147072.67858072917 & val loss = 118924.9609375
epoch 4/3 ===> train loss = 92289.0591796875 & val loss = 70115.6796875
epoch 5/4 ===> train loss = 54898.99424641927 & val loss = 43662.5
epoch 6/5 ===> train loss = 33974.427083333336 & val loss = 26413.89453125
epoch 7/6 ===> train loss = 19684.045532226562 & val loss = 14618.1572265625
epoch 8/7 ===> train loss = 10836.780305989583 & val loss = 8229.2138671875
epoch 9/8 ===> train loss = 6428.206595865885 & val loss = 5149.33447265625
epoch 10/9 ===> train loss = 4113.055594380697 & val loss = 3312.384521484375
epoch 11/10 ===> train loss = 2617.0180976867678 & val loss = 2097.61865234375
epoch 12/11 ===> train loss = 1636.9827852249146 & val loss = 1321.8304443359375
epoch 13/12 ===> train loss = 1035.9868611653646 & val loss = 852.4829711914062
epoch 14/13 ===> t